# 리포트 51 — 수신 → ECA → 거리도플러 → CFAR, 사슬의 형상은 파형이 정한다

> ### 한 일
> **세 조명원을 하나의 동일한 검출 사슬에 물리고, 사슬의 각 단계가 파형마다 어떤 형상(거리 빈 수 · ECA 탭 수 · 도플러 빈)을 갖는지를 표로 고정했다.**

### 결과
1. 사슬은 네 단계다 — 2채널 수신 → ECA 로 직접파 제거 → 거리-도플러 상관 → CA-CFAR 판정. 각 단계는 앞 단계의 잔류물을 물려받는다.
2. 직접파 세기 DNR 은 WiFi 43.0 dB ⟨outputs/verify_eca.json : meta.setups[1].dnr_db⟩ · LTE 60.0 dB ⟨outputs/verify_eca.json : meta.setups[2].dnr_db⟩ · 5G 48.9 dB ⟨outputs/verify_eca.json : meta.setups[0].dnr_db⟩ 다 — 수신단에서 가장 큰 신호이고 2단계가 지울 대상이다.
3. ECA 탭 수는 파형이 정한다 — WiFi 24 ⟨outputs/verify_eca.json : meta.setups[1].n_taps⟩ · LTE 14 ⟨outputs/verify_eca.json : meta.setups[2].n_taps⟩ · 5G 32 ⟨outputs/verify_eca.json : meta.setups[0].n_taps⟩.
4. 도플러 빈은 세 파형 모두 48 ⟨outputs/verify_cfar.json : meta.M_cpi⟩개다(CPI 당 프레임 수). 거리 빈 수와 PRF 는 파형마다 다르고, 아래 표가 그 형상을 한 자리에 모은다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 사슬 구현 | 네 단계가 한 파일에 있다 — `src/passive_process.py:42`(수신) · `:93,124`(ECA) · `:133`(거리-도플러) · `:153`(CA-CFAR) |
| 형상 표 | 선언값을 옮겨 적지 않고 검증 산출물 `outputs/verify_eca.json:meta.setups` 에서 직접 뽑는다 |
| 세 파형 동일 사슬 | 코드 경로가 하나다 — 파형이 바꾸는 것은 형상 파라미터뿐이다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_eca.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_eca.json` |
| 소요 | ECA 검증 · 그림은 각각 수 분 (GPU 1장) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 44 «상시이면서 내용을 미리 아는 신호는 표준마다…»](44_illuminators.ipynb) | 세 표준의 상시 기준신호와 대역 |
| [편 47 «바이스태틱 거리 분해능은 c/B»](47_range-convention.ipynb) | 거리 규약 $\Delta R_b = c/B_{ref}$ |

---

## 수신 신호가 판정이 되기까지

패시브 검출은 네 단계다. 각 단계는 앞 단계의 잔류물을 물려받는다.

| 단계 | 하는 일 | 코드 |
|---|---|---|
| 1. 수신 | 서베일런스(표적 쪽을 보는 채널) + 레퍼런스(조명원을 직접 받는 채널) 2채널 | `src/passive_process.py:42` |
| 2. ECA | 직접파를 서베일런스에서 투영 제거 | `src/passive_process.py:93,124` |
| 3. 거리-도플러(CAF) | 레퍼런스와 지연 · 도플러 상관 | `src/passive_process.py:133` |
| 4. CA-CFAR | 이웃 셀로 문턱을 세우고 판정 | `src/passive_process.py:153` |

![f1_chain](../outputs/figures/report04_f1_chain.png)

**그림 1.** 수신 신호는 어떤 단계를 거쳐 검출 판정이 되는가?

## 사슬의 형상 — 파형이 정하는 것

직접파는 수신단에서 가장 큰 신호다. 그 크기가 DNR 이고, 2단계가 지울 대상이다. 거리 빈 수와 ECA 탭 수는 파형이 정하고, 도플러 빈은 세 파형 모두 48 ⟨outputs/verify_cfar.json : meta.M_cpi⟩개다.

| 파형 | DNR | ECA 탭 | 거리 빈 | PRF | Δf_d |
|---|---|---|---|---|---|
| WiFi 80MHz | 43.0 dB | 24 | 16 | 1000 Hz | 20.83 Hz |
| LTE 20MHz | 60.0 dB | 14 | 6 | 1000 Hz | 20.83 Hz |
| 5G NR 100MHz | 48.9 dB | 32 | 24 | 2000 Hz | 41.67 Hz |

출처 ⟨outputs/verify_eca.json : meta.setups⟩

## 이 사슬 위에서 무엇이 결정되나

2단계의 소거 깊이와 그 대가는 [편 52 «탭을 늘리면 환경이 정한 바닥에서 멈추고»](52_eca.ipynb) 가, 4단계 문턱의 눈금은 [편 53 «운용 형상에서 경험 Pfa 를 재니 명목값의…»](53_cfar-calib.ipynb) 가 든다. 3단계가 만드는 응답의 모양은 [편 49 «검출기가 실제로 쓰는 커널 그대로 모호함수를…»](49_ambiguity.ipynb) 가 이미 쟀다.

세 파형이 같은 코드 경로를 지나므로, 뒤 편들이 재는 격차는 사슬 차이가 아니라 파형 차이다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 회전 블레이드 산란을 이 사슬에 넣는 조건을 세운다 | 마이크로도플러를 검출 판정에 쓰는 조건이 결정된다 | [편 43 «상시 기준신호가 주는 것은 날개끝 확산이 아니…»](43_md-prf.ipynb) |
| 2채널 수신을 실제 X410 캡처로 바꿔 같은 사슬을 돌린다 | 시뮬 사슬과 실측 사슬이 같은 형상 표 위에 선다 | [편 67 «X410 의 12-bit ADC 동적범위가 직…»](67_hardware.ipynb) |